# บทที่ 7: Priority Queue และ Heap> **รายวิชา:** Data Structures and Algorithms (สำหรับนิสิต IT ชั้นปีที่ 2)> **อ้างอิงแนวเนื้อหา:** Narasimha Karumanchi, *Data Structures and Algorithmic Thinking with Python*> **วิธีใช้:** อ่านคำอธิบาย → รันโค้ดทีละเซลล์ → ลองแก้ค่า input → ทำแบบฝึกหัดท้ายบท (เฉลยอยู่ท้ายไฟล์)---

## 7.1 แนวคิด**Heap** = complete binary tree ที่เก็บใน **อาร์เรย์** และรักษา *heap property*- **Min-heap:** `parent <= children` → ค่าน้อยสุดอยู่ที่ index 0- **Max-heap:** `parent >= children`**การแมป index (เริ่มที่ 0)**- ลูกซ้ายของ i = `2*i + 1`, ลูกขวา = `2*i + 2`, พ่อของ i = `(i-1)//2`| Operation | Complexity ||---|---|| `peek` (ค่าน้อย/มากสุด) | \(O(1)\) || `insert` (percolate up) | \(O(\log n)\) || `extract` (percolate down) | \(O(\log n)\) || `build_heap` จากอาร์เรย์ | \(O(n)\) ไม่ใช่ \(O(n\log n)\) |

In [ ]:
class MinHeap:    def __init__(self, data=None):        self.a = list(data) if data else []        if self.a:            self._build()    # ---------- helper ----------    @staticmethod    def _parent(i): return (i - 1) // 2    @staticmethod    def _left(i):   return 2 * i + 1    @staticmethod    def _right(i):  return 2 * i + 2    def _sift_up(self, i):        while i > 0 and self.a[i] < self.a[self._parent(i)]:            p = self._parent(i)            self.a[i], self.a[p] = self.a[p], self.a[i]            i = p    def _sift_down(self, i):        n = len(self.a)        while True:            smallest, l, r = i, self._left(i), self._right(i)            if l < n and self.a[l] < self.a[smallest]: smallest = l            if r < n and self.a[r] < self.a[smallest]: smallest = r            if smallest == i: return            self.a[i], self.a[smallest] = self.a[smallest], self.a[i]            i = smallest    def _build(self):        # sift_down จากโหนดที่ไม่ใช่ leaf ตัวสุดท้ายย้อนขึ้นไป -> O(n)        for i in range(len(self.a) // 2 - 1, -1, -1):            self._sift_down(i)    # ---------- public ----------    def push(self, x):        self.a.append(x); self._sift_up(len(self.a) - 1)     # O(log n)    def pop(self):        if not self.a: raise IndexError('heap is empty')        top = self.a[0]        last = self.a.pop()        if self.a:            self.a[0] = last            self._sift_down(0)        return top                                            # O(log n)    def peek(self): return self.a[0]    def __len__(self): return len(self.a)h = MinHeap([9, 4, 7, 1, 8, 2])print("heap array:", h.a, "| min =", h.peek())h.push(0)print("หลัง push 0:", h.a)print("เรียงออกมา:", [h.pop() for _ in range(len(h))])       # heap sort โดยปริยาย

In [ ]:
# ใช้งานจริง: โมดูล heapq (เป็น min-heap)import heapqa = [9, 4, 7, 1, 8, 2]heapq.heapify(a)                     # O(n)print("heapified:", a)heapq.heappush(a, 0)print("pop เล็กสุด:", heapq.heappop(a), "->", a)# เทคนิคทำ MAX-heap: ใส่ค่าติดลบmx = []for x in [9, 4, 7, 1]:    heapq.heappush(mx, -x)print("มากสุด:", -heapq.heappop(mx))# priority queue ที่มี tuple (priority, counter, task) - counter กันปัญหาเทียบ task ไม่ได้import itertoolscounter = itertools.count()pq = []for pri, task in [(2, 'เขียนรายงาน'), (1, 'ตรวจการบ้าน'), (3, 'ประชุม')]:    heapq.heappush(pq, (pri, next(counter), task))while pq:    pri, _, task = heapq.heappop(pq)    print(f"  ทำลำดับความสำคัญ {pri}: {task}")

In [ ]:
# การประยุกต์ใช้ heapimport heapq# 1) k ค่าที่ใหญ่ที่สุด ด้วย min-heap ขนาด k -> O(n log k) ดีกว่า sort ทั้งชุดเมื่อ k << ndef k_largest(a, k):    h = a[:k]    heapq.heapify(h)    for x in a[k:]:        if x > h[0]:            heapq.heapreplace(h, x)      # pop + push ในครั้งเดียว    return sorted(h, reverse=True)# 2) รวม k sorted lists -> O(N log k)def merge_k_sorted(lists):    h = [(l[0], i, 0) for i, l in enumerate(lists) if l]    heapq.heapify(h)    out = []    while h:        val, li, idx = heapq.heappop(h)        out.append(val)        if idx + 1 < len(lists[li]):            heapq.heappush(h, (lists[li][idx+1], li, idx+1))    return out# 3) หา median แบบสตรีม ด้วย 2 heaps -> O(log n) ต่อค่าclass MedianFinder:    def __init__(self):        self.lo, self.hi = [], []        # lo = max-heap (เก็บค่าติดลบ), hi = min-heap    def add(self, x):        heapq.heappush(self.lo, -x)        heapq.heappush(self.hi, -heapq.heappop(self.lo))        if len(self.hi) > len(self.lo):            heapq.heappush(self.lo, -heapq.heappop(self.hi))    def median(self):        if len(self.lo) > len(self.hi): return -self.lo[0]        return (-self.lo[0] + self.hi[0]) / 2print("3 ตัวใหญ่สุด:", k_largest([5, 12, 1, 9, 22, 3, 17], 3))print("merge:", merge_k_sorted([[1,4,9],[2,5],[3,7,8]]))mf = MedianFinder()for x in [5, 15, 1, 3]:    mf.add(x); print("  add", x, "-> median =", mf.median())

---## แบบฝึกหัดบทที่ 71. เขียน heap sort ให้สมบูรณ์ โดยใช้ max-heap และเรียง in-place2. เขียน `MaxHeap` เต็มคลาส (ไม่ใช้เทคนิคค่าติดลบ)3. หาองค์ประกอบที่ใหญ่อันดับ k จากอาร์เรย์ที่ไม่เรียง4. ตรวจว่าอาร์เรย์ที่ให้มาเป็น min-heap ที่ถูกต้องหรือไม่

### เฉลยบทที่ 7

In [ ]:
# ข้อ 1: heap sort in-place -> O(n log n) time, O(1) spacedef heap_sort(a):    n = len(a)    def sift_down(i, size):        while True:            largest, l, r = i, 2*i+1, 2*i+2            if l < size and a[l] > a[largest]: largest = l            if r < size and a[r] > a[largest]: largest = r            if largest == i: return            a[i], a[largest] = a[largest], a[i]            i = largest    for i in range(n // 2 - 1, -1, -1):     # build max-heap : O(n)        sift_down(i, n)    for end in range(n - 1, 0, -1):         # สลับ max ไปท้าย แล้วซ่อมกอง        a[0], a[end] = a[end], a[0]        sift_down(0, end)    return a# ข้อ 2class MaxHeap:    def __init__(self): self.a = []    def push(self, x):        self.a.append(x); i = len(self.a) - 1        while i > 0 and self.a[i] > self.a[(i-1)//2]:            p = (i-1)//2            self.a[i], self.a[p] = self.a[p], self.a[i]; i = p    def pop(self):        top, last = self.a[0], self.a.pop()        if self.a:            self.a[0] = last; i, n = 0, len(self.a)            while True:                big, l, r = i, 2*i+1, 2*i+2                if l < n and self.a[l] > self.a[big]: big = l                if r < n and self.a[r] > self.a[big]: big = r                if big == i: break                self.a[i], self.a[big] = self.a[big], self.a[i]; i = big        return top# ข้อ 3import heapqdef kth_largest(a, k):    return heapq.nlargest(k, a)[-1]# ข้อ 4def is_min_heap(a):    n = len(a)    return all(a[i] <= a[c]               for i in range(n // 2)               for c in (2*i+1, 2*i+2) if c < n)print(heap_sort([9, 4, 7, 1, 8, 2]))mh = MaxHeap()for x in [3, 9, 5]: mh.push(x)print("max =", mh.pop())print("อันดับ 2 ใหญ่สุด:", kth_largest([5, 12, 1, 22, 17], 2))print(is_min_heap([1, 3, 2, 7, 8]), is_min_heap([3, 1, 2]))